In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as f
from  torchvision import transforms as t,datasets
from torch.utils.data import DataLoader
import time
from tqdm import tqdm
d=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
import pandas as pd


ModuleNotFoundError: No module named 'torch'

In [3]:
#hyperparameters
batch_size=64
lr=0.001
epochs=100


In [ ]:
class self_att(nn.Module):
    def __init__(self,embed_size=128,heads=12):
        super(self_att,self).__init__()
        self.embed_size=embed_size
        self.head=heads
        self.head_dim=embed_size//self.head
        self.qvk=nn.Linear(embed_size,embed_size*3,bias=False)
        self.fc_out=nn.Linear(embed_size,embed_size)
    def forward(self,x):
        B,L,_=x.shape
        out=self.qvk(x)
        chunks = torch.chunk(out, chunks=3, dim=-1)
        q=chunks[0]
        k=chunks[1]
        v=chunks[2]
        q = q.view(B, L, self.head, self.head_dim).transpose(1, 2)
        k = k.view(B, L, self.head, self.head_dim).transpose(1, 2)
        v = v.view(B, L, self.head, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        a=attn = torch.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, L, self.embed_size)
        out=self.fc_out(out)
        return out
class masked_self_att(nn.Module):
        def __init__(self,embed_size=128,heads=12):
            super().__init__()
            self.embed_size=embed_size
            self.head=heads
            self.head_dim=embed_size//self.heads
            self.qvk=nn.Linear(embed_size,embed_size*3,bias=False)
            self.fc_out=nn.Linear(embed_size,embed_size)
        def forward(self,x):
            B,L,_=x.shape
            out=self.qvk(x)
            chunks = torch.chunk(out, chunks=3, dim=-1)
            q=chunks[0]
            k=chunks[1]
            v=chunks[2]
            q = q.view(B, L, self.head, self.head_dim).transpose(1, 2)
            k = k.view(B, L, self.head, self.head_dim).transpose(1, 2)
            v = v.view(B, L, self.head, self.head_dim).transpose(1, 2)
            out = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, is_causal=True
                )
            out = out.transpose(1, 2).contiguous().view(B, L, self.embed_size)
            out=self.fc_out(out)
            return out
class cross_attention(nn.Module):
        def __init__(self,embed_size=128,heads=12):
            super().__init__()
            self.embed_size=embed_size
            self.head=heads
            self.head_dim=embed_size//self.head
            self.q=nn.Linear(embed_size,embed_size,bias=False)
            self.kv=nn.Linear(embed_size,embed_size*2,bias=False)
            self.fc_out=nn.Linear(embed_size,embed_size)
        def forward(self,x,y):
            B,L,_=x.shape
            out=self.q(x)
            out_d=self.kv(y)
            chunks = torch.chunk(out_d, chunks=2, dim=-1)
            k=chunks[0]
            v=chunks[1]
            q=self.q(x)
            q = q.view(B, L, self.head, self.head_dim).transpose(1, 2)
            k = k.view(B, L, self.head, self.head_dim).transpose(1, 2)
            v = v.view(B, L, self.head, self.head_dim).transpose(1, 2)
            out = torch.nn.functional.scaled_dot_product_attention(
                q, k, v,
                )
            out = out.transpose(1, 2).contiguous().view(B, L, self.embed_size)
            out=self.fc_out(out)
            return out

In [ ]:
#encoder
class encoder_block(nn.Module):
    def __init__(self,embedding,head):
        super().__init__()
        self.s_a=self_att(embedding,head)
        self.layer_norm=nn.LayerNorm(embedding)
        self.layer_norm2=nn.LayerNorm(embedding)
        self.layer_1=nn.Linear(embedding,embedding*2)
        self.layer_2=nn.Linear(embedding*2,embedding)
        self.dropout=nn.Dropout(0.5)
    def forward(self,x):
        out=self.s_a(x)
        out=out+x
        out=self.layer_norm(out)
        out_nn=self.layer_1(out)
        out_nn=self.dropout(out_nn)
        out_nn=nn.functional.relu(out_nn)
        out_nn=self.layer_2(out_nn)
        out_nn=self.dropout(out_nn)
        out=out+out_nn
        out=self.layer_norm2(out)
        return out
class DecoderBlock(nn.Module):
    def __init__(self, embedding, head):
        super().__init__()
        self.m_s_a = masked_self_att(embedding, head)
        self.c_s_a = cross_attention(embedding, head)
        self.ln1 = nn.LayerNorm(embedding)
        self.ln2 = nn.LayerNorm(embedding)
        self.ln3 = nn.LayerNorm(embedding)
        self.ff1 = nn.Linear(embedding, embedding * 4)
        self.ff2 = nn.Linear(embedding * 4, embedding)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, enc_out):
        attn1 = self.dropout(self.m_s_a(x))
        x = self.ln1(x + attn1)
        attn2 = self.dropout(self.c_s_a(x, enc_out))
        x = self.ln2(x + attn2)
        ff = self.ff2(torch.relu(self.ff1(x)))
        ff = self.dropout(ff)
        x = self.ln3(x + ff)
        return x

In [ ]:
class encoder_layer(nn.Module):
    def __init__(self,embedding,head,layers):
        super().__init__()
        self.layers=nn.ModuleList([encoder_block(embedding,head) for _ in range(layers)])
    def forward(self,x):
        for layer in self.layers:
            x=layer(x)
        return x
class decoder_layer(nn.Module):
    def __init__(self,embedding,head,layers):
        super().__init__()
        self.layers=nn.ModuleList([DecoderBlock(embedding,head) for _ in range(layers)])
    def forward(self,x,enc_out):
        for layer in self.layers:
            x=layer(x,enc_out)
        return x
class transformer(nn.Module):
    def __init__(self,src_vocab_size,trg_vocab_size,embedding,head,enc_layers,dec_layers):
        super().__init__()
        self.src_embedding=nn.Embedding(src_vocab_size,embedding)
        self.trg_embedding=nn.Embedding(trg_vocab_size,embedding)
        self.encoder=encoder_layer(embedding,head,enc_layers)
        self.decoder=decoder_layer(embedding,head,dec_layers)
        self.fc_out=nn.Linear(embedding,2)
    def forward(self,src,trg):
        enc_out=self.encoder(self.src_embedding(src))
        dec_out=self.decoder(self.trg_embedding(trg),enc_out)
        out=self.fc_out(dec_out)
        return out
